# From events to labels

The core pipeline: **collect** events into the raw buffer → **map** them as an activity
graph → **conclude labels** for what is happening.

Data goes to a throwaway directory, so this notebook leaves no trace in `data/`.

In [ ]:
from gitrecon.config import Config, load_env

load_env()  # GH_TOKEN / HF_TOKEN from dotenv files, never overriding your shell

In [ ]:
import tempfile
from pathlib import Path

from gitrecon.sources.github_api import GitHubClient
from gitrecon.storage import RawBuffer

config = Config(data_dir=Path(tempfile.mkdtemp(prefix="gitrecon-")))
client = GitHubClient(config=config)
buffer = RawBuffer(root=config.raw_dir)

## 1. Listening: an Events API feed

`Feed.parse()` takes `public`, `user:NAME`, `org:NAME` or `repo:OWNER/NAME`.
`EventPoller.poll()` returns only events not seen before; with `state_dir` the ETag and
seen ids survive between runs (`listen()` keeps polling at GitHub's `X-Poll-Interval`).

In [ ]:
from gitrecon.sources.events import EventPoller, Feed

poller = EventPoller(client=client, feed=Feed.parse("public"), state_dir=config.state_dir)
batch = poller.poll()
buffer.append("events", batch, suffix=poller.feed.name)
len(batch), poller.state.poll_interval

## 2. History: one hour of GH Archive

Every public GitHub event since 2011, one gzip file per hour, streamed straight into the
buffer partition. Older hours are small; recent ones are hundreds of MB.

In [ ]:
from gitrecon.sources import gharchive

hour = gharchive.parse_hour("2015-01-01-15")
path = gharchive.download_hour(hour, buffer.partition("gharchive", hour), config)
f"{path.stat().st_size / 2**20:.1f} MiB"

## 3. Mapping: `ActivityGraph`

Nodes are entities (`user:`, `repo:`, `org:`, `gist:`), edges are relations evidenced by
event ids.

In [ ]:
from gitrecon.mapping import ActivityGraph

graph = ActivityGraph()
graph.ingest(buffer.read())
summary = graph.summary()
summary["nodes"], summary["hubs"][:5]

In [ ]:
hub = summary["hubs"][0][0]
graph.neighbors(hub)[:10]

## 4. Concluding: `Labeler` and `Thresholds`

Each rule looks at one entity's activity and may conclude a label with a confidence and
the evidence (event ids). Thresholds are plain dataclass fields - tune them per question.

In [ ]:
from datetime import timedelta

from gitrecon.analysis import Labeler, Thresholds

labeler = Labeler(thresholds=Thresholds(window=timedelta(hours=1), burst_events=100, push_flood_commits=500))
labeler.ingest(buffer.read())
labels = labeler.run()
for label in labels[:15]:
    print(label)

In [ ]:
from collections import Counter

Counter(label.name for label in labels)

A label carries its reasons:

In [ ]:
interesting = next(label for label in labels if label.name != "declared-bot")
interesting.details, interesting.evidence[:5]

CLI equivalents: `gitrecon events public --save`, `gitrecon archive 2015-01-01-15`,
`gitrecon map`, `gitrecon label`. Writing your own rule: see the docs guide
*Writing a label rule* (`task manuals -- writing-a-label-rule`).